# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Signal 1: GSC clicks

Verdict: CONFIRMED

GSC clicks show observable search activity. Most content has zero clicks, while a smaller number of items show measurable activity. This signal is useful for identifying content with search engagement.

In [29]:
# Signal 1: GSC clicks bucket check

df["click_bucket"] = pd.cut(
    df["gsc_clicks"],
    bins=[-1, 0, 5, float("inf")],
    labels=["zero", "low", "high"]
)

click_table = df["click_bucket"].value_counts().reset_index()
click_table.columns = ["bucket", "n"]

click_table

,bucket,n
0,zero,940
1,low,59
2,high,1


Signal 2: GA4 sessions

Verdict: FALSE

GA4 sessions were not useful in this sample because all observed rows had zero sessions. This signal could not provide ranking information for the baseline rule.

In [30]:
# Signal 2: GA4 sessions bucket check

df["session_bucket"] = pd.cut(
    df["ga4_sessions"],
    bins=[-1, 0, 10, float("inf")],
    labels=["zero", "low", "high"]
)

session_table = df["session_bucket"].value_counts().reset_index()
session_table.columns = ["bucket", "n"]

session_table

,bucket,n
0,zero,1000
1,low,0
2,high,0


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [31]:
from huggingface_hub import login
login()

In [32]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    streaming=True,
    split="train"
)

df = pd.DataFrame(ds.take(1000))

df.head()

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,0


In [33]:
import os

# Create baseline action score
df["baseline_score"] = (
    df["gsc_clicks"].fillna(0) * 0.4 +
    df["ga4_sessions"].fillna(0) * 0.3 +
    df["scroll_events"].fillna(0) * 0.2 +
    df["ga4_engaged_sessions"].fillna(0) * 0.1
)

# Rank items
df["rank"] = df["baseline_score"].rank(
    ascending=False,
    method="first"
)

# Add reason codes
def reason(row):
    if row["gsc_clicks"] > 0:
        return "GSC_ACTIVITY"
    elif row["ga4_sessions"] > 0:
        return "GA4_ACTIVITY"
    else:
        return "LOW_ACTIVITY"

df["reason_code"] = df.apply(reason, axis=1)
df["action"] = "REVIEW_CONTENT"
# Save output
os.makedirs("work/outputs", exist_ok=True)

df.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

df.head()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,baseline_score,rank,reason_code,action
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0.0,61.0,LOW_ACTIVITY,REVIEW_CONTENT
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0.0,62.0,LOW_ACTIVITY,REVIEW_CONTENT
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0.0,63.0,LOW_ACTIVITY,REVIEW_CONTENT
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0.0,64.0,LOW_ACTIVITY,REVIEW_CONTENT
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0.0,65.0,LOW_ACTIVITY,REVIEW_CONTENT


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top-20 Review:

The ranked items were selected using the baseline score built from observed engagement signals.

Action:
REVIEW_CONTENT

Why they are selected:
Items with measurable GSC activity received higher priority in the ranking queue.

What would make it wrong:
The ranking could be incorrect if tracking data is incomplete, signals are missing, or the observed activity does not represent future content value.

In [34]:
# Top-20 review

top20 = df.sort_values(
    by="rank",
    ascending=True
).head(20)

top20[[
    "rank",
    "content_hash_id",
    "action",
    "baseline_score",
    "reason_code",
    "gsc_clicks",
    "ga4_sessions",
    "scroll_events"
]]

,rank,content_hash_id,action,baseline_score,reason_code,gsc_clicks,ga4_sessions,scroll_events
146,1.0,content_f94fe855380e150f,REVIEW_CONTENT,2.4,GSC_ACTIVITY,6,0,0
766,2.0,content_f94fe855380e150f,REVIEW_CONTENT,2.0,GSC_ACTIVITY,5,0,0
953,3.0,content_37b3bafd5f88fdd1,REVIEW_CONTENT,1.6,GSC_ACTIVITY,4,0,0
446,4.0,content_f94fe855380e150f,REVIEW_CONTENT,1.2,GSC_ACTIVITY,3,0,0
675,5.0,content_48b01ea98ec1ea03,REVIEW_CONTENT,0.8,GSC_ACTIVITY,2,0,0
18,6.0,content_d237e43c93144a57,REVIEW_CONTENT,0.4,GSC_ACTIVITY,1,0,0
41,7.0,content_4b90d8f71a8d9c59,REVIEW_CONTENT,0.4,GSC_ACTIVITY,1,0,0
53,8.0,content_67c0464f614c0606,REVIEW_CONTENT,0.4,GSC_ACTIVITY,1,0,0
54,9.0,content_c0cc958ff5d50e85,REVIEW_CONTENT,0.4,GSC_ACTIVITY,1,0,0
63,10.0,content_0642dc7f62d4f780,REVIEW_CONTENT,0.4,GSC_ACTIVITY,1,0,0


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks:

The lowest-ranked picks have a baseline score of 0 because no observed engagement signals were available (GSC clicks and GA4 sessions were both zero).

These picks may be weak because:
- The content may have low activity or limited tracking data.
- Missing data availability can reduce the confidence of the score.
- Additional signals would be needed before making a final decision.

Leakage check:

I checked the available columns and confirmed that the score only uses observed historical activity signals. No future performance windows or product flags were used in the baseline score.

In [35]:
# Weak picks
weak_picks = df.sort_values(
    by="baseline_score",
    ascending=True
).head(10)

weak_picks[[
    "content_hash_id",
    "baseline_score",
    "reason_code",
    "gsc_clicks",
    "ga4_sessions"
]]

,content_hash_id,baseline_score,reason_code,gsc_clicks,ga4_sessions
655,content_975c9759f98eb0c0,0.0,LOW_ACTIVITY,0,0
640,content_eebfb2913c3a0eaf,0.0,LOW_ACTIVITY,0,0
641,content_63c7a1498f76797f,0.0,LOW_ACTIVITY,0,0
642,content_519c6ac7943e296b,0.0,LOW_ACTIVITY,0,0
643,content_aa8aa2fb530f0af0,0.0,LOW_ACTIVITY,0,0
644,content_d0d52c7ff7217dac,0.0,LOW_ACTIVITY,0,0
645,content_4cfee21dfadec4af,0.0,LOW_ACTIVITY,0,0
646,content_75ab9996eb32bf9b,0.0,LOW_ACTIVITY,0,0
647,content_2cbccd216580ad42,0.0,LOW_ACTIVITY,0,0
648,content_167aff5fed6593c7,0.0,LOW_ACTIVITY,0,0


In [36]:
# Check for possible leakage columns
leak_columns = [
    col for col in df.columns
    if "future" in col.lower()
    or "next" in col.lower()
    or "product" in col.lower()
]

leak_columns

[]

#  Output check

In [37]:
import os
os.listdir("work/outputs")

['baseline_action_score.csv']

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.